In [86]:
import pandas as pd
import numpy as np
from pathlib import Path

In [87]:
def find_files(folder: Path, extension: str) -> list[Path]:
    return [f for f in folder.iterdir() if f.is_file() and f.suffix == extension]

def convertAndDownsample(fileData: np.lib.npyio.NpzFile, delay: float) -> pd.DataFrame:
    tripTime = 1 # The trip event happened at 1 second
    # keys ['t', 'freq', 'measured_delay', 'last_cmd_time', 'buses', 'per_bus_delay']
    t = fileData["t"] - tripTime
    freq = fileData["freq"]
    # Add a feature relative to the number of setpoints received up to now (count the number of elements in measured_delay that are less than or equal to the current time)
    measured_delay = fileData["per_bus_delay"]
    setpoint_count = np.array([np.sum(measured_delay <= time) if np.sum(measured_delay <= time) > 0 else 0 for time in t])
    # Add a feature for the delay of the average delay of the commands up to the current time
    average_delay = np.array([np.mean(measured_delay[measured_delay <= time]) if np.sum(measured_delay <= time) > 0 else 0 for time in t])

    # Compute lag features using timestamp lookups.
    freq_lag_50ms = np.zeros(len(t))
    freq_lag_100ms = np.zeros(len(t))

    idx_50 = np.searchsorted(t, t - 0.05, side="right") - 1
    valid_50 = idx_50 >= 0
    freq_lag_50ms[valid_50] = freq[idx_50[valid_50]]

    idx_100 = np.searchsorted(t, t - 0.1, side="right") - 1
    valid_100 = idx_100 >= 0
    freq_lag_100ms[valid_100] = freq[idx_100[valid_100]]

    # Downsample the data by selecting only the samples that are at least `delay` seconds apart
    downsampledIndices = [0]  # Always include the first sample
    lastTime = t[0]
    for i in range(1, len(t)):
        if t[i] - lastTime >= delay:
            downsampledIndices.append(i)
            lastTime = t[i]
    return pd.DataFrame({
        "t-from-trip": t[downsampledIndices],
        "freq": freq[downsampledIndices],
        "freq-lag-50ms": freq_lag_50ms[downsampledIndices],
        "freq-lag-100ms": freq_lag_100ms[downsampledIndices],
        "setpoint-count": setpoint_count[downsampledIndices],
        "average-delay": average_delay[downsampledIndices]
    })


In [88]:
dataFolder = Path("./data").absolute()
resultsFolder = Path("./results").absolute()
classToFolders = {
    "direct": ["case_iec104_direct_percmd"],
    "delay": ["case_iec104_mitm_delay_dmin0p4_dmax0p6_percmd", "case_iec104_mitm_delay_dmin0p15_dmax0p35_percmd", "case_iec104_mitm_delay_dmin0p65_dmax0p85_percmd"],
    "drop": ["case_iec104_mitm_drop_pdrop0p1_percmd", "case_iec104_mitm_drop_pdrop0p5_percmd", "case_iec104_mitm_drop_pdrop0p9_percmd"],
    "mod": ["case_iec104_mitm_mod_pmod0p1_f1p3_percmd", "case_iec104_mitm_mod_pmod0p5_f1p3_percmd", "case_iec104_mitm_mod_pmod0p9_f1p3_percmd"]
}
classToFoldersPaths = {c: [resultsFolder / f for f in folders] for c, folders in classToFolders.items()}
# Find all files with ".npz" extension in the folders and associate it with the class
classToFilesPaths = {c: [f for folder in folders for f in find_files(folder, ".npz")] for c, folders in classToFoldersPaths.items()}

In [89]:
# Load the data from the .npz files and create a DataFrame
classToData = {}

for c, files in classToFilesPaths.items():
    classDataDf = [] 
    for filePath in files:
        df = np.load(filePath)
        classDataDf.append(df)
    classToData[c] = classDataDf

In [90]:
# Create the final dataframe

finalDf = pd.DataFrame(columns=["t-from-trip", "freq", "freq-lag-50ms", "freq-lag-100ms", "setpoint-count", "average-delay", "run-id", "class"])
runId = 0
for c, dataList in classToData.items():
    for data in dataList:
        runId += 1
        dataDf = convertAndDownsample(data, delay=0.0001)
        dataDf["class"] = c
        dataDf["run-id"] = runId
        # Remove the rows where t-from-trip is negative (before the trip event)
        dataDf = dataDf[dataDf["t-from-trip"] >= 0]
        finalDf = pd.concat([finalDf, dataDf[["t-from-trip", "freq", "freq-lag-50ms", "freq-lag-100ms", "setpoint-count", "average-delay", "run-id", "class"]]], ignore_index=True)

/tmp/ipykernel_167138/74697246.py:13: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  finalDf = pd.concat([finalDf, dataDf[["t-from-trip", "freq", "freq-lag-50ms", "freq-lag-100ms", "setpoint-count", "average-delay", "run-id", "class"]]], ignore_index=True)


In [94]:
finalDf.head(n=10)

,t-from-trip,freq,freq-lag-50ms,freq-lag-100ms,setpoint-count,average-delay,run-id,class
0,0.000100,59.999947,60.000000,60.000000,0,0.000000,1,direct
1,0.033433,59.969277,60.000000,60.000000,0,0.000000,1,direct
2,0.066767,59.946259,59.999947,60.000000,0,0.000000,1,direct
3,0.099900,59.927761,59.969277,60.000000,0,0.000000,1,direct
4,0.100100,59.927659,59.969277,59.999947,0,0.000000,1,direct
5,0.100644,59.927382,59.969277,59.999947,1,0.100644,1,direct
6,0.101190,59.927116,59.969277,59.999947,2,0.100917,1,direct
7,0.101690,59.926951,59.969277,59.999947,3,0.101175,1,direct
8,0.102162,59.926914,59.969277,59.999947,4,0.101421,1,direct
9,0.102632,59.926961,59.969277,59.999947,5,0.101664,1,direct


In [95]:
finalDf.to_csv(dataFolder / "final_data.csv", index=False)